In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import plotly.graph_objects as go
import math

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

def load_dataset(filename):
    BASE_DIR = './data'
    EXTENSION = '.csv'
    TARGET_PATH = f'{BASE_DIR}/{filename}{EXTENSION}'
    try:
        return pd.read_csv(TARGET_PATH)
    except FileNotFoundError:
        return None
    except UnicodeDecodeError:
        return pd.read_csv(TARGET_PATH, encoding='euc-kr')

df = load_dataset('크루즈입항')

In [6]:
# ============================================================
# 1. 라이브러리 불러오기
# ============================================================

import numpy as np
import pandas as pd
import plotly.graph_objects as go


# ============================================================
# 2. 데이터 확인 및 전처리
# ============================================================

if df is None:
    raise ValueError("df가 None입니다. 데이터 로딩을 먼저 실행해주세요.")

if not isinstance(df, pd.DataFrame):
    raise TypeError("df가 pandas DataFrame이 아닙니다.")

frequency_source = df.copy()
frequency_source.columns = frequency_source.columns.str.strip()

required_columns = ["입출", "선박용도", "입항일시"]

missing_columns = [
    col for col in required_columns
    if col not in frequency_source.columns
]

if missing_columns:
    raise ValueError(f"필요한 컬럼이 없습니다: {missing_columns}")


# ============================================================
# 3. 크루즈 입항 데이터 추출
# ============================================================

monthly_data = frequency_source[
    (frequency_source["입출"] == "입항") &
    (frequency_source["선박용도"] == "크루즈선")
].copy()

monthly_data["입항일시"] = pd.to_datetime(
    monthly_data["입항일시"],
    errors="coerce"
)

# 분석 기간: 2013.01.01 ~ 2026.10.16
monthly_data = monthly_data[
    (monthly_data["입항일시"] >= "2013-01-01") &
    (monthly_data["입항일시"] < "2026-10-17")
].copy()

if monthly_data.empty:
    raise ValueError("분석 기간에 해당하는 크루즈 입항 데이터가 없습니다.")


# 연도와 월 추출
monthly_data["연도"] = monthly_data["입항일시"].dt.year
monthly_data["월"] = monthly_data["입항일시"].dt.month


# ============================================================
# 4. 연도별 월별 입항 건수 집계
# ============================================================

year_range = range(2013, 2027)
month_range = range(1, 13)

# 연도별·월별 입항 건수
monthly_counts = (
    monthly_data
    .groupby(["연도", "월"])
    .size()
)

# 전체 연도와 월 조합 생성
# 기록이 없는 연월은 0건으로 처리
full_index = pd.MultiIndex.from_product(
    [year_range, month_range],
    names=["연도", "월"]
)

monthly_counts = monthly_counts.reindex(
    full_index,
    fill_value=0
)


# ============================================================
# 5. 월별 평균 입항 건수 계산
# ============================================================

monthly_average = []

for month in month_range:

    # 1~10월: 2013~2026년 (14개 연도)
    # 11~12월: 2013~2025년 (13개 연도)
    if month <= 10:
        valid_years = range(2013, 2027)
    else:
        valid_years = range(2013, 2026)

    # 해당 월의 연도별 입항 건수 추출
    yearly_counts = (
        monthly_counts
        .xs(month, level="월")
        .reindex(valid_years, fill_value=0)
    )

    # 연도별 평균 계산
    average_count = yearly_counts.mean()

    monthly_average.append({
        "월": month,
        "평균입항건수": average_count
    })


# 그래프용 DataFrame 생성
chart_data = pd.DataFrame(monthly_average)

chart_data["월표시"] = (
    chart_data["월"].astype(str) + "월"
)


# ============================================================
# 6. 그래프 색상 및 데이터 라벨 설정
# ============================================================

max_value = chart_data["평균입항건수"].max()

# 최댓값: 주황 / 나머지: 파랑
colors = [
    "#E67E22" if value == max_value
    else "#3478C0"
    for value in chart_data["평균입항건수"]
]

# 막대 위에 표시할 값
labels = [
    f"{value:.1f}"
    for value in chart_data["평균입항건수"]
]


# ============================================================
# 7. 그래프 생성
# ============================================================

fig_monthly_average = go.Figure()

fig_monthly_average.add_trace(
    go.Bar(
        x=chart_data["월표시"],
        y=chart_data["평균입항건수"],

        marker_color=colors,
        marker_line_width=0,

        text=labels,
        textposition="outside",

        textfont=dict(
            size=25,
            color="#243247"
        ),

        cliponaxis=False,

        customdata=chart_data[["월"]].to_numpy(),

        hovertemplate=(
            "<b>%{x}</b>"
            "<br>월별 평균 입항 건수: %{y:.1f}건"
            "<extra></extra>"
        )
    )
)


# ============================================================
# 8. 그래프 레이아웃
# ============================================================

fig_monthly_average.update_layout(

    title=dict(
        text=(
            "<b>부산항 크루즈 월별 평균 입항건수</b>"
            "<br>"
            "<span style='font-size:23px;color:#657487'>"
            "월별 입항 집중 시기 분석"
            "</span>"
        ),

        x=0.5,
        y=0.94,
        xanchor="center",

        font=dict(
            size=30,
            color="#243247"
        )
    ),

    template="plotly_white",

    width=1650,
    height=900,

    paper_bgcolor="white",
    plot_bgcolor="white",

    font=dict(
        family="Malgun Gothic, Arial, sans-serif",
        size=18,
        color="#243247"
    ),

    margin=dict(
        l=125,
        r=65,
        t=155,
        b=145
    ),

    bargap=0.32,

    showlegend=False
)


# ============================================================
# 9. X축 설정
# ============================================================

fig_monthly_average.update_xaxes(

    title=None,

    categoryorder="array",
    categoryarray=chart_data["월표시"].tolist(),

    showgrid=False,

    tickfont=dict(
        size=20,
        color="#243247"
    )
)


# ============================================================
# 10. Y축 설정
# ============================================================

# 최댓값에 맞춰 2단위 간격의 상한 설정
y_max = max(
    18,
    int(np.ceil(max_value / 2) * 2 + 2)
)

fig_monthly_average.update_yaxes(

    title=dict(
        text="월별 평균 입항건수 (건)",
        standoff=22,
        font=dict(size=22)
    ),

    range=[0, y_max],

    dtick=2,

    showgrid=True,
    gridcolor="#E4EAF1",
    gridwidth=1,

    zeroline=False,

    tickfont=dict(
        size=19,
        color="#243247"
    )
)


# ============================================================
# 11. 그래프 하단 주석
# ============================================================

fig_monthly_average.add_annotation(

    x=0,
    y=-0.20,

    xref="paper",
    yref="paper",

    text=(
        "※ 분석 기간: 2013.01.01~2026.10.16"
        "<br>※ 1~10월은 14개 연도, 11~12월은 13개 연도 평균"
        "<br>※ 기록이 없는 연월은 0건으로 처리"
    ),

    showarrow=False,

    xanchor="left",
    align="left",

    font=dict(
        size=17,
        color="#657487"
    )
)


# ============================================================
# 12. 그래프 출력 및 이미지 저장
# ============================================================

fig_monthly_average.show(
    config={
        "displaylogo": False,

        "toImageButtonOptions": {
            "format": "png",
            "filename": "PPT_부산항_크루즈_월별_평균_입항건수",
            "width": 1650,
            "height": 900,
            "scale": 3
        }
    }
)
